In [1]:
import pandas as pd
from rdkit import Chem
def get_reactive_atom_indices(smiles):
    # 解析 SMILES
    mol = Chem.MolFromSmiles(smiles)
    
    # 获取标记为反应位点的原子
    reactive_atoms = []
    for atom in mol.GetAtoms():
        # 检查是否带有反应位点标记
        if atom.HasProp('molAtomMapNumber'):
            reactive_atoms.append(atom.GetIdx())  # 获取原子序号
    
    return reactive_atoms

In [2]:
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize

def standardize_smiles_with_atom_map(smiles: str) -> str:
    try:
        # 将 SMILES 转换为分子对象
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            raise ValueError("Invalid SMILES string.")
        
        # 提取位点信息（Atom Maps）
        atom_map = {atom.GetIdx(): atom.GetAtomMapNum() for atom in mol.GetAtoms() if atom.GetAtomMapNum() > 0}
        
        # 标准化分子（使用 Standardizer）
        uncharger = rdMolStandardize.Uncharger()  # 去质子化
        mol = uncharger.uncharge(mol)
        
        # 去除多余氢原子
        mol = Chem.RemoveHs(mol)
        
        # 恢复位点信息（Atom Maps）
        for idx, map_num in atom_map.items():
            mol.GetAtomWithIdx(idx).SetAtomMapNum(map_num)
        
        # 返回标准化后的 SMILES
        standardized_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
        return standardized_smiles
    except Exception as e:
        return f"Error: {e}"

In [3]:
df = pd.read_csv('predict_site.csv')
substrates = df['SMILES'].to_list()
predicts = df['predict_site'].to_list()
predicts = [i.split('|') for i in predicts]
atomidxs = []
for i in predicts:
    atomidxs_part = []
    for j in i:
        try:
            j = standardize_smiles_with_atom_map(j)
            atomidx = get_reactive_atom_indices(j)
            atomidxs_part.append(atomidx)
        except:
            pass
    atomidxs.append(atomidxs_part)
df['atomidxs'] = atomidxs
df

[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Running Uncharger
[07:24:19] Run

,SMILES,metabolite,predict_site,atomidxs
0,C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...,C#C[C@]1(O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)CC[C@@...,CC(=O)O[C@@]1(C#C)CC[C@@H]2[C@]1(CC)CC[C@@H]1[...,"[[17], [15], [6], [12], [4], [12], [4], [25], ..."
1,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)OC2O[C@H](...,CC([CH3:1])(O)C1=CC=CC=C1CC[C@@H](SCC1(CC(=O)O...,"[[40], [37, 38], [19], [14], [32], [40], [21],..."
2,CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2)cc1,CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2O)cc1|N...,CC(C)CNCC1=CC=C(C2=CC=CC=C2S(=O)(=O)N2CCC[CH2:...,"[[23], [5], [1], [23], [5], [25], [22], [3], [..."
3,CC(CN1c2ccccc2Sc2ccccc21)N(C)C,CC(CN1c2ccccc2S(=O)c2ccccc21)N(C)C|CNC(C)CN1c2...,CC(CN1C2=CC=CC=C2[S:1]C2=CC=CC=C12)N(C)C|CC(CN...,"[[10], [10], [13], [10], [15], [14], [14, 15],..."
4,CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NC...,CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NC...,CC1(C)CCC(CN2CCN(C3=CC=C(C(=O)NS(=O)(=O)C4=CC=...,"[[26], [6], [34], [2], [15], [40], [60], [60],..."
...,...,...,...,...
60,O=C(O)c1cc(O)c2c(c1)C(C1c3cc(C(=O)O)cc(O)c3C(=...,O=C(O)c1cc(O)c2c(c1)C(C1c3cccc(O)c3C(=O)c3c(O)...,C1=CC=C2C(=C1OC1C(O)C(O)C(O)C(CO)O1)C(=O)C1=C(...,"[[27], [27], [27], [27], [27], [29], [27], [27..."
61,O=C1CN=C(c2ccccc2)c2cc(Cl)ccc2N1,O=C(O)[C@H]1OC(OC2CN=C(c3ccccc3)c3cc(Cl)ccc3N2...,O=C1[CH2:1]N=C(C2=CC=CC=C2)C2=CC(Cl)=CC=C2N1|O...,"[[18], [15], [4], [8], [1], [13], [16], [18], ..."
62,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=NC1O,O=C(O)[C@H]1O[C@@H](OC2N=C(c3ccccc3)c3cc(Cl)cc...,O=C1NC2=CC=C(Cl)C=C2C(C2=CC=CC=C2)=NC1[OH:1]|C...,"[[19], [10], [19], [14], [19], [10], [1], [19]..."
63,O=P1(NCCCl)OCCCN1CCCl,O=CCCl|NP1(=O)OCCCN1CCCl|O=P1(NCCCl)OCCC(O)N1C...,O=P1(NCCCl)OCC[CH2:1]N1CCCl|O=P1(NCCCl)OCCCN1[...,"[[9], [11], [9], [9], [9], [9], [7], [4], [8],..."


In [4]:
df.to_pickle('ourmodel_results.pickle')

In [5]:
df = pd.read_pickle('ourmodel_results.pickle')
df

,SMILES,metabolite,predict_site,atomidxs
0,C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...,C#C[C@]1(O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)CC[C@@...,CC(=O)O[C@@]1(C#C)CC[C@@H]2[C@]1(CC)CC[C@@H]1[...,"[[17], [15], [6], [12], [4], [12], [4], [25], ..."
1,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)OC2O[C@H](...,CC([CH3:1])(O)C1=CC=CC=C1CC[C@@H](SCC1(CC(=O)O...,"[[40], [37, 38], [19], [14], [32], [40], [21],..."
2,CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2)cc1,CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2O)cc1|N...,CC(C)CNCC1=CC=C(C2=CC=CC=C2S(=O)(=O)N2CCC[CH2:...,"[[23], [5], [1], [23], [5], [25], [22], [3], [..."
3,CC(CN1c2ccccc2Sc2ccccc21)N(C)C,CC(CN1c2ccccc2S(=O)c2ccccc21)N(C)C|CNC(C)CN1c2...,CC(CN1C2=CC=CC=C2[S:1]C2=CC=CC=C12)N(C)C|CC(CN...,"[[10], [10], [13], [10], [15], [14], [14, 15],..."
4,CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NC...,CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NC...,CC1(C)CCC(CN2CCN(C3=CC=C(C(=O)NS(=O)(=O)C4=CC=...,"[[26], [6], [34], [2], [15], [40], [60], [60],..."
...,...,...,...,...
60,O=C(O)c1cc(O)c2c(c1)C(C1c3cc(C(=O)O)cc(O)c3C(=...,O=C(O)c1cc(O)c2c(c1)C(C1c3cccc(O)c3C(=O)c3c(O)...,C1=CC=C2C(=C1OC1C(O)C(O)C(O)C(CO)O1)C(=O)C1=C(...,"[[27], [27], [27], [27], [27], [29], [27], [27..."
61,O=C1CN=C(c2ccccc2)c2cc(Cl)ccc2N1,O=C(O)[C@H]1OC(OC2CN=C(c3ccccc3)c3cc(Cl)ccc3N2...,O=C1[CH2:1]N=C(C2=CC=CC=C2)C2=CC(Cl)=CC=C2N1|O...,"[[18], [15], [4], [8], [1], [13], [16], [18], ..."
62,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=NC1O,O=C(O)[C@H]1O[C@@H](OC2N=C(c3ccccc3)c3cc(Cl)cc...,O=C1NC2=CC=C(Cl)C=C2C(C2=CC=CC=C2)=NC1[OH:1]|C...,"[[19], [10], [19], [14], [19], [10], [1], [19]..."
63,O=P1(NCCCl)OCCCN1CCCl,O=CCCl|NP1(=O)OCCCN1CCCl|O=P1(NCCCl)OCCC(O)N1C...,O=P1(NCCCl)OCC[CH2:1]N1CCCl|O=P1(NCCCl)OCCCN1[...,"[[9], [11], [9], [9], [9], [9], [7], [4], [8],..."


In [6]:
subs = df['SMILES'].to_list()
atomidxs = df['atomidxs'].to_list()
dict_sub2atomidxs = dict(zip(subs,atomidxs))
dict_sub2atomidxs

{'C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)CC[C@@H]4[C@H]3CC[C@@]21CC': [[17],
  [15],
  [6],
  [12],
  [4],
  [12],
  [4],
  [25],
  [26],
  [21],
  [4, 21],
  [12]],
 'CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cccc(/C=C/c2ccc3ccc(Cl)cc3n2)c1': [[40],
  [37, 38],
  [19],
  [14],
  [32],
  [40],
  [21],
  [12],
  [8],
  [40],
  [38, 39],
  [21],
  [7]],
 'CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2)cc1': [[23],
  [5],
  [1],
  [23],
  [5],
  [25],
  [22],
  [3],
  [4],
  [16],
  [4],
  [4],
  [4],
  [25],
  [25]],
 'CC(CN1c2ccccc2Sc2ccccc21)N(C)C': [[10],
  [10],
  [13],
  [10],
  [15],
  [14],
  [14, 15],
  [17],
  [17],
  [5],
  [13],
  [15, 16],
  [16],
  [13]],
 'CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NCC5CCOCC5)c([N+](=O)[O-])c4)c(Oc4cnc5[nH]ccc5c4)c3)CC2)=C(c2ccc(Cl)cc2)C1': [[26],
  [6],
  [34],
  [2],
  [15],
  [40],
  [60],
  [60],
  [31],
  [32],
  [59],
  [60],
  [18],
  [34],
  [50],
  [51],
  [7],
  [7],
  [45, 46]],
 'CC1(C)S[C@@H]2[C@H](NC(=O)[C@H](N)c3ccc(O)cc3)

In [7]:
df_total = pd.read_pickle('/home/datahouse1/raojingxin/myprojects/enzyme/metabolism/SOM_cases/output_65_site_truth_correct_withatomidx_noenzyme.pickle')
substrates = df_total['substrates'].to_list()
ourmodel_results_old = [dict_sub2atomidxs[i] for i in substrates]
df_total['ourmodel_results_old'] = ourmodel_results_old
df_total

,substrates,true_sites,true_sites_old,gnn_results,smartcyp_results,xenosite_results,somp_results,metapredictor_results_old,ourmodel_results_old
0,C#C[C@]1(OC(C)=O)CC[C@H]2[C@@H]3CCC4=C/C(=N/O)...,[3],[3],"{0: 9, 1: 6, 2: 2, 3: 1, 4: 4, 5: 18, 6: 13, 7...","{0: 53, 1: 66, 2: 72, 3: 69, 4: 60, 5: 5, 6: 5...","{0: 65, 1: 107, 2: 138, 3: 77, 4: 107, 5: 74, ...","{0: 97, 1: 156, 2: 140, 3: 143, 4: 93, 5: 93, ...","[[3], [17], [15, 17], [12], [15], [13, 14], [1...","[[17], [15], [6], [12], [4], [12], [4], [25], ..."
1,CC(C)(O)c1ccccc1CC[C@@H](SCC1(CC(=O)O)CC1)c1cc...,"[0, 10, 19, 13]","[40, 32, 10, 19]","{0: 18, 1: 5, 2: 16, 3: 3, 4: 34, 5: 36, 6: 42...","{0: 36, 1: 83, 2: 36, 3: 75, 4: 89, 5: 44, 6: ...","{0: 68, 1: 204, 2: 75, 3: 36, 4: 184, 5: 112, ...","{0: 94, 1: 112, 2: 100, 3: 178, 4: 221, 5: 98,...","[[6], [19], [40], [7], [1, 40], [40], [1], [40...","[[40], [37, 38], [19], [14], [32], [40], [21],..."
2,CC(C)CNCc1ccc(-c2ccccc2S(=O)(=O)N2CCCC2)cc1,"[1, 2, 4, 5, 22, 23]","[1, 4, 5, 22, 23, 25]","{0: 15, 1: 22, 2: 16, 3: 7, 4: 2, 5: 6, 6: 23,...","{0: 26, 1: 21, 2: 26, 3: 6, 4: 18, 5: 13, 6: 5...","{0: 46, 1: 63, 2: 53, 3: 74, 4: 30, 5: 73, 6: ...","{0: 70, 1: 32, 2: 76, 3: 91, 4: 127, 5: 41, 6:...","[[23], [22], [25], [4], [1], [3], [3, 25], [13...","[[23], [5], [1], [23], [5], [25], [22], [3], [..."
3,CC(CN1c2ccccc2Sc2ccccc21)N(C)C,"[17, 10, 14]","[10, 11, 12, 13, 14, 15, 16, 17]","{0: 12, 1: 6, 2: 4, 3: 5, 4: 16, 5: 13, 6: 11,...","{0: 26, 1: 11, 2: 18, 3: 30, 4: 39, 5: 24, 6: ...","{0: 56, 1: 58, 2: 64, 3: 22, 4: 108, 5: 65, 6:...","{0: 78, 1: 67, 2: 92, 3: 96, 4: 107, 5: 61, 6:...","[[13], [14], [10], [7], [11, 12, 13, 14, 15, 1...","[[10], [10], [13], [10], [15], [14], [14, 15],..."
4,CC1(C)CCC(CN2CCN(c3ccc(C(=O)NS(=O)(=O)c4ccc(NC...,"[2, 7, 50, 59, 60]","[2, 7, 50, 59, 60]","{0: 34, 1: 42, 2: 32, 3: 45, 4: 29, 5: 20, 6: ...","{0: 68, 1: 108, 2: 68, 3: 65, 4: 42, 5: 126, 6...","{0: 120, 1: 289, 2: 127, 3: 125, 4: 149, 5: 21...","{0: 126, 1: 214, 2: 132, 3: 54, 4: 97, 5: 160,...","[[58], [2], [59], [45], [60], [25], [39], [34]...","[[26], [6], [34], [2], [15], [40], [60], [60],..."
...,...,...,...,...,...,...,...,...,...
60,O=C(O)c1cc(O)c2c(c1)C(C1c3cc(C(=O)O)cc(O)c3C(=...,"[26, 47]","[30, 47]","{0: 48, 1: 7, 2: 28, 3: 9, 4: 14, 5: 1, 6: 2, ...","{0: 64, 1: 49, 2: 64, 3: 59, 4: 30, 5: 56, 6: ...","{0: 238, 1: 229, 2: 128, 3: 268, 4: 161, 5: 28...","{0: 208, 1: 162, 2: 276, 3: 272, 4: 115, 5: 20...","[[30], [11], [10, 30], [30, 47], [11, 30], [47...","[[27], [27], [27], [27], [27], [29], [27], [27..."
61,O=C1CN=C(c2ccccc2)c2cc(Cl)ccc2N1,"[0, 1, 2]","[16, 17, 18]","{0: 3, 1: 1, 2: 5, 3: 2, 4: 6, 5: 18, 6: 9, 7:...","{0: 30, 1: 33, 2: 3, 3: 8, 4: 48, 5: 42, 6: 19...","{0: 50, 1: 84, 2: 50, 3: 19, 4: 74, 5: 92, 6: ...","{0: 52, 1: 85, 2: 13, 3: 54, 4: 101, 5: 71, 6:...","[[8], [16], [15], [18], [18], [9], [16, 17, 18...","[[18], [15], [4], [8], [1], [13], [16], [18], ..."
62,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=NC1O,"[0, 1, 19]","[16, 17, 18, 19]","{0: 6, 1: 5, 2: 4, 3: 10, 4: 18, 5: 13, 6: 9, ...","{0: 30, 1: 33, 2: 27, 3: 42, 4: 20, 5: 11, 6: ...","{0: 55, 1: 90, 2: 20, 3: 103, 4: 60, 5: 39, 6:...","{0: 54, 1: 94, 2: 95, 3: 97, 4: 55, 5: 35, 6: ...","[[19], [14], [18], [15], [19], [18, 19], [5]]","[[19], [10], [19], [14], [19], [10], [1], [19]..."
63,O=P1(NCCCl)OCCCN1CCCl,"[2, 9, 10, 11]","[2, 4, 9, 10, 12, 13]","{0: 6, 1: 8, 2: 1, 3: 2, 4: 11, 5: 12, 6: 13, ...","{0: 30, 1: 33, 2: 15, 3: 6, 4: 24, 6: 30, 7: 1...","{0: 54, 1: 39, 2: 23, 3: 46, 4: 41, 5: 70, 6: ...","{0: 49, 1: 75, 2: 55, 3: 24, 4: 61, 5: 39, 6: ...","[[2], [10], [12], [9], [4], [12, 13], [5], [2,...","[[9], [11], [9], [9], [9], [9], [7], [4], [8],..."


In [8]:
df_total.to_pickle('/home/datahouse1/raojingxin/myprojects/enzyme/metabolism/SOM_cases/output_65_site_truth_correct_withatomidx_noenzyme.pickle')

In [3]:
import pandas as pd
df_total = pd.read_pickle('/home/datahouse1/raojingxin/myprojects/enzyme/metabolism/SOM_cases/output_30_site_truth_correct_withatomidx_noenzyme.pickle')

In [4]:
df_total

,name,substrate,predict,top_n,sites,gnn_results,smartcyp_results,xenosite_results,somp_results
0,1,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,2,[30],"{0: 20, 1: 3, 2: 7, 3: 11, 4: 19, 5: 8, 6: 1, ...","{0: 61, 1: 66, 2: 42, 3: 23, 4: 72, 5: 37, 6: ...","{0: 13, 1: 36, 2: 14, 3: 15, 4: 29, 5: 3, 6: 3...","{0: 153, 1: 116, 2: 171, 3: 84, 4: 46, 5: 120,..."
1,2,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-...,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(O)c(-c...,3,"[28, 37]","{0: 41, 1: 37, 2: 12, 3: 1, 4: 27, 5: 3, 6: 17...","{1: 75, 2: 28, 3: 90, 4: 95, 5: 43, 6: 89, 7: ...","{0: 50, 1: 54, 2: 76, 3: 123, 4: 94, 5: 22, 6:...","{0: 216, 1: 196, 2: 112, 3: 227, 4: 217, 5: 61..."
2,3,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21 0.57|...,1,[20],"{0: 5, 1: 4, 2: 9, 3: 11, 4: 14, 5: 23, 6: 3, ...","{0: 27, 1: 8, 2: 20, 3: 33, 4: 24, 5: 59, 6: 3...","{0: 31, 1: 34, 2: 47, 3: 49, 4: 60, 5: 109, 6:...","{0: 16, 1: 13, 2: 36, 3: 31, 4: 29, 5: 113, 6:..."
3,4,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12 0.83|...,1,[3],"{0: 18, 1: 4, 2: 16, 3: 3, 4: 21, 5: 17, 6: 22...","{0: 5, 1: 51, 2: 48, 3: 36, 4: 60, 5: 24, 6: 3...","{0: 59, 1: 103, 2: 71, 3: 23, 4: 138, 5: 61, 6...","{0: 64, 1: 97, 2: 108, 3: 61, 4: 146, 5: 51, 6..."
4,5,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)...,2,[3],"{0: 17, 1: 21, 2: 18, 3: 9, 4: 8, 5: 2, 6: 13,...","{0: 10, 1: 51, 2: 30, 3: 6, 4: 45, 5: 3, 6: 11...","{0: 55, 1: 104, 2: 63, 3: 30, 4: 116, 5: 12, 6...","{0: 14, 1: 104, 2: 81, 3: 40, 4: 121, 5: 41, 6..."
5,6,Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@...,O=C([C@@H]1CC1(F)F)N1[C@H]2CC[C@@H]1CN(c1ccnc(...,4,[4],"{0: 1, 1: 3, 2: 25, 3: 22, 4: 2, 5: 28, 6: 12,...","{0: 3, 1: 38, 2: 26, 3: 60, 4: 24, 5: 66, 6: 3...","{0: 62, 1: 46, 2: 72, 3: 128, 4: 29, 5: 133, 6...","{0: 20, 1: 73, 2: 102, 3: 119, 4: 135, 5: 177,..."
6,7,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3c[nH]c4c...,2,[40],"{0: 30, 1: 38, 2: 19, 3: 43, 4: 13, 5: 33, 6: ...","{0: 4, 1: 51, 2: 97, 3: 90, 4: 69, 5: 111, 6: ...","{0: 52, 1: 73, 2: 167, 3: 118, 4: 37, 5: 198, ...","{0: 86, 1: 102, 2: 119, 3: 164, 4: 139, 5: 211..."
7,8,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]...,OCc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H...,3,"[12, 13]","{0: 24, 1: 31, 2: 27, 3: 21, 4: 20, 5: 22, 6: ...","{0: 21, 1: 68, 2: 40, 3: 41, 4: 75, 5: 18, 6: ...","{0: 82, 1: 143, 2: 83, 3: 103, 4: 156, 5: 84, ...","{0: 27, 1: 153, 2: 79, 3: 99, 4: 161, 5: 145, ..."
8,10,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1ccnc2ccccc12,Nc1ccsc1C(=O)Nc1ccc(OC(F)(F)F)cc1 0.73|O=C(Nc1...,2,[23],"{0: 10, 1: 1, 2: 7, 3: 30, 4: 18, 5: 11, 6: 14...","{0: 76, 1: 67, 2: 45, 3: 66, 4: 41, 5: 28, 6: ...","{0: 80, 1: 102, 2: 27, 3: 140, 4: 77, 5: 51, 6...","{0: 106, 1: 92, 2: 97, 3: 128, 4: 46, 5: 73, 6..."
9,11,NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2ncnc3[...,NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)[...,1,[8],"{0: 12, 1: 6, 2: 2, 3: 10, 4: 4, 5: 13, 6: 16,...","{0: 21, 1: 75, 2: 71, 3: 70, 4: 45, 5: 23, 6: ...","{0: 44, 1: 148, 2: 119, 3: 87, 4: 45, 5: 94, 6...","{0: 64, 1: 122, 2: 91, 3: 98, 4: 148, 5: 45, 6..."


In [6]:
gnn_results = df_total['gnn_results'].to_list()
smartcyp_results = df_total['smartcyp_results'].to_list()
xenosite_results = df_total['xenosite_results'].to_list()
somp_results = df_total['somp_results'].to_list()

gnn_sorted_atoms = [[k for k, v in sorted(i.items(), key=lambda item: item[1])] for i in gnn_results]
smartcyp_sorted_atoms = [[k for k, v in sorted(i.items(), key=lambda item: item[1])] for i in smartcyp_results]
# xenosite_sorted_atoms = [[k for k, v in sorted(i.items(), key=lambda item: item[1])] for i in xenosite_results]
xenosite_sorted_atoms = []
for i in xenosite_results:
    try:
        sub_result = [k for k, v in sorted(i.items(), key=lambda item: item[1])]
    except:
        sub_result = ['None']
    xenosite_sorted_atoms.append(sub_result)
somp_sorted_atoms = []
for i in somp_results:
    try:
        sub_result = [k for k, v in sorted(i.items(), key=lambda item: item[1])]
    except:
        sub_result = ['None']
    somp_sorted_atoms.append(sub_result)
# somp_sorted_atoms = [[k for k, v in sorted(i.items(), key=lambda item: item[1])] for i in somp_results]

df_total['gnn_results'] = gnn_sorted_atoms
df_total['smartcyp_results'] = smartcyp_sorted_atoms
df_total['xenosite_results'] = xenosite_sorted_atoms
df_total['somp_results'] = somp_sorted_atoms
df_total

,name,substrate,predict,top_n,sites,gnn_results,smartcyp_results,xenosite_results,somp_results
0,1,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,CC(C)(C)[C@H](NC(=O)C(F)(F)F)C(=O)N1CC2(C[C@H]...,2,[30],"[6, 23, 1, 32, 35, 34, 2, 5, 24, 33, 3, 13, 27...","[17, 12, 15, 24, 13, 14, 20, 23, 3, 22, 25, 21...","[29, 7, 5, 31, 21, 20, 17, 32, 35, 13, 14, 15,...","[13, 16, 6, 4, 11, 7, 19, 8, 23, 26, 28, 3, 12..."
1,2,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(OC)c(-...,CC[C@@H](C(=O)Nc1ccc(C(N)=O)c(F)c1)n1cc(O)c(-c...,3,"[28, 37]","[3, 29, 5, 10, 40, 7, 37, 39, 32, 34, 24, 2, 1...","[11, 35, 26, 29, 23, 17, 12, 8, 2, 16, 21, 34,...","[5, 11, 0, 16, 1, 8, 19, 7, 2, 20, 26, 12, 40,...","[11, 13, 28, 27, 5, 12, 15, 17, 30, 31, 36, 35..."
2,3,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(C)CC21,CCCCCc1cc(O)c2c(c1)OC(C)(C)C1CCC(CO)CC21 0.57|...,1,[20],"[7, 8, 6, 1, 0, 14, 13, 15, 2, 11, 3, 19, 21, ...","[18, 1, 17, 22, 19, 2, 4, 21, 0, 16, 20, 3, 6,...","[0, 1, 8, 2, 3, 19, 12, 20, 18, 4, 14, 6, 16, ...","[1, 0, 4, 3, 2, 20, 18, 17, 19, 14, 15, 6, 21,..."
3,4,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)c1,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12 0.83|...,1,[3],"[10, 12, 3, 1, 13, 16, 22, 11, 14, 23, 20, 17,...","[0, 8, 21, 22, 13, 18, 20, 17, 5, 23, 16, 3, 6...","[3, 12, 18, 20, 21, 22, 0, 5, 14, 8, 25, 23, 2...","[8, 18, 22, 14, 23, 19, 5, 21, 3, 0, 13, 17, 2..."
4,5,Cc1ccc(N)cc1C(=O)N[C@H](C)c1cccc2ccccc12,CC(=O)Nc1ccc(C)c(C(=O)N[C@H](C)c2cccc3ccccc23)...,2,[3],"[10, 5, 8, 11, 20, 18, 21, 4, 3, 9, 19, 14, 6,...","[5, 3, 0, 6, 19, 20, 11, 16, 18, 15, 2, 21, 14...","[5, 10, 3, 16, 19, 18, 6, 20, 0, 12, 2, 9, 11,...","[0, 16, 20, 3, 5, 12, 21, 17, 19, 15, 11, 22, ..."
5,6,Cn1cc(Nc2nccc(N3C[C@H]4CC[C@@H](C3)N4C(=O)[C@@...,O=C([C@@H]1CC1(F)F)N1[C@H]2CC[C@@H]1CN(c1ccnc(...,4,[4],"[0, 4, 1, 18, 19, 16, 20, 11, 24, 22, 15, 6, 1...","[0, 11, 16, 12, 15, 21, 20, 10, 26, 4, 2, 13, ...","[4, 10, 17, 1, 13, 21, 27, 14, 0, 2, 11, 19, 6...","[0, 13, 17, 23, 14, 15, 24, 26, 12, 27, 16, 1,..."
6,7,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3cn(C)c4c...,C=CC(=O)Nc1cc(Nc2ncc(C(=O)OC(C)C)c(-c3c[nH]c4c...,2,[40],"[13, 40, 41, 15, 42, 37, 36, 23, 22, 33, 16, 3...","[0, 41, 42, 37, 23, 26, 25, 39, 38, 16, 33, 27...","[8, 4, 36, 40, 0, 1, 22, 26, 17, 18, 25, 32, 2...","[41, 42, 23, 37, 33, 16, 38, 40, 39, 17, 27, 1..."
7,8,Cc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]...,OCc1ccc([C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H...,3,"[12, 13]","[30, 15, 21, 29, 24, 14, 13, 9, 19, 12, 16, 25...","[8, 10, 7, 12, 14, 18, 5, 0, 30, 25, 28, 21, 2...","[9, 13, 11, 15, 10, 24, 29, 12, 14, 25, 7, 18,...","[35, 0, 10, 6, 9, 30, 13, 33, 16, 19, 2, 15, 1..."
8,10,O=C(Nc1ccc(OC(F)(F)F)cc1)c1sccc1NCc1ccnc2ccccc12,Nc1ccsc1C(=O)Nc1ccc(OC(F)(F)F)cc1 0.73|O=C(Nc1...,2,[23],"[1, 28, 29, 19, 22, 26, 2, 27, 12, 0, 5, 24, 7...","[17, 25, 16, 21, 24, 23, 29, 30, 26, 5, 12, 28...","[19, 2, 24, 27, 5, 15, 12, 26, 29, 7, 4, 22, 0...","[17, 30, 25, 4, 24, 13, 29, 8, 5, 12, 26, 9, 2..."
9,11,NC1(C(=O)N[C@@H](CCO)c2ccc(Cl)cc2)CCN(c2ncnc3[...,NC1(C(=O)N[C@@H](CCOC2O[C@H](C(=O)O)[C@@H](O)[...,1,[8],"[8, 2, 24, 4, 7, 1, 15, 25, 18, 3, 10, 0, 5, 1...","[26, 17, 28, 25, 7, 22, 0, 20, 5, 18, 11, 14, ...","[8, 0, 24, 4, 18, 7, 10, 11, 14, 15, 16, 26, 6...","[8, 6, 5, 17, 12, 30, 15, 0, 23, 18, 7, 29, 11..."


In [7]:
df_total.to_pickle('/home/datahouse1/raojingxin/myprojects/enzyme/metabolism/SOM_cases/output_30_site_truth_correct_withrank_noenzyme.pickle')

In [8]:
def get_unique_numbers(nums, n):
    """
    从列表中提取前 n 个不同的数字。

    参数:
        nums (list): 输入的数字列表。
        n (int): 需要提取的不同数字的数量。

    返回:
        list: 包含前 n 个不同数字的列表。
    """
    unique_nums = []
    seen = set()

    for num in nums:
        if num not in seen:
            unique_nums.append(num)
            seen.add(num)
        if len(unique_nums) == n:
            break

    return unique_nums

# 示例使用
nums = [1, 2, 2, 3, 4, 3, 5, 1, 6]
n = 4
result = get_unique_numbers(nums, n)
print(result)  # 输出: [1, 2, 3, 4]


[1, 2, 3, 4]


In [9]:
df_total = pd.read_pickle('/home/datahouse1/raojingxin/myprojects/enzyme/metabolism/SOM_cases/output_30_site_truth_correct_withrank_noenzyme.pickle')

# metapredictor_results_old = df_total['metapredictor_results_old'].to_list()
# ourmodel_results_old = df_total['ourmodel_results_old'].to_list()
# ourmodel_results_new = []
# for i in ourmodel_results_old:
#     list_part = []
#     for j in i:
#         list_part = list_part + j
#     ourmodel_results_new.append(list_part)
# metapredictor_results_new = []
# for i in metapredictor_results_old:
#     list_part = []
#     for j in i:
#         list_part = list_part + j
#     metapredictor_results_new.append(list_part)

# true_sites_old = df_total['true_sites_old'].to_list()
true_sites = df_total['sites'].to_list()

def acc_top_n(n,modelresults,true_sites):
    modelresults = [i[:n] for i in modelresults]
    total_num = len(true_sites)
    correct_num = 0
    for idx,i in enumerate(true_sites):
        modelresult = modelresults[idx]
        modelresult = set(modelresult)
        i = set(i)
        intersection = i & modelresult
        if len(intersection) > 0:
            correct_num = correct_num + 1
    acc = correct_num/total_num
    print(f'top {n} accuracy = {acc}')
    return acc

# def acc_top_n_old(n,modelresults,true_sites):
#     total_num = len(true_sites)
#     correct_num = 0
#     for idx,i in enumerate(true_sites):
#         modelresult = modelresults[idx]
#         modelresult = get_unique_numbers(modelresult, n)
#         modelresult = set(modelresult)
#         i = set(i)
#         intersection = i & modelresult
#         if len(intersection) > 0:
#             correct_num = correct_num + 1
#     acc = correct_num/total_num
#     print(f'top {n} accuracy = {acc}')
#     return acc

gnn_top1_acc = acc_top_n(1,gnn_sorted_atoms,true_sites)
gnn_top3_acc = acc_top_n(3,gnn_sorted_atoms,true_sites)
gnn_top5_acc = acc_top_n(5,gnn_sorted_atoms,true_sites)
gnn_topn = []
gnn_topn.append(gnn_top1_acc)
gnn_topn.append(gnn_top3_acc)
gnn_topn.append(gnn_top5_acc)

smartcyp_top1_acc = acc_top_n(1,smartcyp_sorted_atoms,true_sites)
smartcyp_top3_acc = acc_top_n(3,smartcyp_sorted_atoms,true_sites)
smartcyp_top5_acc = acc_top_n(5,smartcyp_sorted_atoms,true_sites)
smartcyp_topn = []
smartcyp_topn.append(smartcyp_top1_acc)
smartcyp_topn.append(smartcyp_top3_acc)
smartcyp_topn.append(smartcyp_top5_acc)

xenosite_top1_acc = acc_top_n(1,xenosite_sorted_atoms,true_sites)
xenosite_top3_acc = acc_top_n(3,xenosite_sorted_atoms,true_sites)
xenosite_top5_acc = acc_top_n(5,xenosite_sorted_atoms,true_sites)
xenosite_topn = []
xenosite_topn.append(xenosite_top1_acc)
xenosite_topn.append(xenosite_top3_acc)
xenosite_topn.append(xenosite_top5_acc)

somp_top1_acc = acc_top_n(1,somp_sorted_atoms,true_sites)
somp_top3_acc = acc_top_n(3,somp_sorted_atoms,true_sites)
somp_top5_acc = acc_top_n(5,somp_sorted_atoms,true_sites)
somp_topn = []
somp_topn.append(somp_top1_acc)
somp_topn.append(somp_top3_acc)
somp_topn.append(somp_top5_acc)

# metapredictor_top1_acc = acc_top_n_old(1,metapredictor_results_new,true_sites_old)
# metapredictor_top3_acc = acc_top_n_old(3,metapredictor_results_new,true_sites_old)
# metapredictor_top5_acc = acc_top_n_old(5,metapredictor_results_new,true_sites_old)
# metapredictor_topn = []
# metapredictor_topn.append(metapredictor_top1_acc)
# metapredictor_topn.append(metapredictor_top3_acc)
# metapredictor_topn.append(metapredictor_top5_acc)

# ourmodel_top1_acc = acc_top_n_old(1,ourmodel_results_new,true_sites_old)
# ourmodel_top3_acc = acc_top_n_old(3,ourmodel_results_new,true_sites_old)
# ourmodel_top5_acc = acc_top_n_old(5,ourmodel_results_new,true_sites_old)
# ourmodel_topn = []
# ourmodel_topn.append(ourmodel_top1_acc)
# ourmodel_topn.append(ourmodel_top3_acc)
# ourmodel_topn.append(ourmodel_top5_acc)

top 1 accuracy = 0.13821138211382114
top 3 accuracy = 0.43089430894308944
top 5 accuracy = 0.6097560975609756
top 1 accuracy = 0.10569105691056911
top 3 accuracy = 0.3008130081300813
top 5 accuracy = 0.45528455284552843
top 1 accuracy = 0.2682926829268293
top 3 accuracy = 0.5040650406504065
top 5 accuracy = 0.6341463414634146
top 1 accuracy = 0.15447154471544716
top 3 accuracy = 0.35772357723577236
top 5 accuracy = 0.43089430894308944


In [10]:
df = pd.DataFrame()
df['top_n'] = ['top1_recall','top3_recall','top5_recall']
df['gnnsom_topn'] = gnn_topn
df['smartcyp_topn'] = smartcyp_topn
df['xenosite_topn'] = xenosite_topn
df['somp_topn'] = somp_topn
# df['metapredictor_topn'] = metapredictor_topn
# df['ourmodel_topn'] = ourmodel_topn
df

,top_n,gnnsom_topn,smartcyp_topn,xenosite_topn,somp_topn
0,top1_recall,0.138211,0.105691,0.268293,0.154472
1,top3_recall,0.430894,0.300813,0.504065,0.357724
2,top5_recall,0.609756,0.455285,0.634146,0.430894
